In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import os
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from joblib import Parallel, delayed

DATA_PATH = "Data/crsp_ff3_daily_1995_2024.parquet"
VAL_DATES_PATH = "Data/val_dates_hyperparam.csv"

features = ["mktrf", "smb", "hml"]
target = "excess_ret"

TEST_START, TEST_END = "2024-01-01", "2024-12-31"

# chosen hyperparams (from the sweep)
BEST_PARAMS = {
  "hidden_dim": 32,
  "lr": 0.001,
  "weight_decay": 0.0005,
  "epochs": 200
}

N_JOBS = max(1, (os.cpu_count() or 8) // 2)

df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

# Load the exact validation dates used for tuning
val_dates = pd.read_csv(VAL_DATES_PATH)["date"]
val_dates = pd.to_datetime(val_dates)

# Test set (2024)
test = df[(df["date"] >= TEST_START) & (df["date"] <= TEST_END)].copy()

# Pre-2024 pool
pre = df[df["date"] <= "2023-12-31"].copy()

# TRAIN = pre minus those validation dates (strict approach)
train_strict = pre[~pre["date"].isin(val_dates)].copy()

print("Pre-2024:", pre.shape)
print("Train (strict, excl val dates):", train_strict.shape)
print("Test 2024:", test.shape)
print("Unique val dates loaded:", len(val_dates))
print("N_JOBS:", N_JOBS)


  2%|▏         | 66/2716 [00:20<01:50, 23.96it/s]

Pre-2024: (13064388, 11)
Train (strict, excl val dates): (12612270, 11)
Test 2024: (684286, 11)
Unique val dates loaded: 252
N_JOBS: 6


In [5]:
class ShallowNN(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)

def fit_ff3_ols(stock_train: pd.DataFrame):
    X = sm.add_constant(stock_train[features])
    y = stock_train[target]
    return sm.OLS(y, X).fit()

def train_nn_train(stock_train: pd.DataFrame, params: dict):
    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(stock_train[features].values)

    # NO target scaling
    y_train = stock_train[target].values.astype(np.float32)  # shape (n,)

    model = ShallowNN(input_dim=X_train.shape[1], hidden_dim=int(params["hidden_dim"]))
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(params["lr"]),
        weight_decay=float(params["weight_decay"])
    )
    loss_fn = nn.MSELoss()

    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    epochs = int(params["epochs"])
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_t)
        loss = loss_fn(preds, y_t)
        loss.backward()
        optimizer.step()

    return model, x_scaler

def process_one_permno(permno: int, train_df: pd.DataFrame, test_df: pd.DataFrame, params: dict):
    stock_train = train_df[train_df["permno"] == permno]
    stock_test  = test_df[test_df["permno"] == permno]

    if stock_test.empty or len(stock_train) < 300:
        return None

    # FF3
    ff3_model = fit_ff3_ols(stock_train)
    X_test_ff3 = sm.add_constant(stock_test[features])
    ff3_pred = ff3_model.predict(X_test_ff3).values

    # NN
    nn_model, x_scaler = train_nn_train(stock_train, params)

    X_test_nn = x_scaler.transform(stock_test[features].values)
    X_test_nn_t = torch.tensor(X_test_nn, dtype=torch.float32)

    nn_model.eval()
    with torch.no_grad():
        nn_pred = nn_model(X_test_nn_t).numpy().flatten()  # already in target units


    # XAI: Integrated Gradients on test set
    # baseline = 0 vector in standardized space
    attrs = integrated_gradients(
        nn_model,
        X_test_nn_t,
        baseline=torch.zeros_like(X_test_nn_t),
        steps=50
    )
    attrs_np = attrs.detach().numpy()  # (T,3)

    mean_abs_attr = np.mean(np.abs(attrs_np), axis=0)      # (3,)
    mean_signed_attr = np.mean(attrs_np, axis=0)           # (3,)

    xai_row = pd.DataFrame([{
        "permno": permno,
        "ig_abs_mktrf": float(mean_abs_attr[0]),
        "ig_abs_smb":   float(mean_abs_attr[1]),
        "ig_abs_hml":   float(mean_abs_attr[2]),
        "ig_mktrf":     float(mean_signed_attr[0]),
        "ig_smb":       float(mean_signed_attr[1]),
        "ig_hml":       float(mean_signed_attr[2]),
        "n_test_obs":   int(len(stock_test))
    }])

    # predictions output 
    out = stock_test[["date", "permno", target]].copy()
    out["ff3_pred"] = ff3_pred
    out["nn_pred"] = nn_pred

    return out, xai_row

#Helper function for xai bit
def integrated_gradients(model, x, baseline=None, steps=50):
    """
    Integrated Gradients for a PyTorch model.
    x: torch.Tensor (n, d) in standardized feature space
    baseline: torch.Tensor (n, d) or None -> zeros
    returns: torch.Tensor (n, d)
    """
    model.eval()
    model.zero_grad(set_to_none=True)

    if baseline is None:
        baseline = torch.zeros_like(x)

    # interpolate between baseline and x
    alphas = torch.linspace(0.0, 1.0, steps, device=x.device).view(-1, 1, 1)
    x_exp = x.unsqueeze(0)          # (1, n, d)
    base_exp = baseline.unsqueeze(0)
    x_interp = base_exp + alphas * (x_exp - base_exp)  # (steps, n, d)

    x_interp.requires_grad_(True)

    y = model(x_interp.reshape(-1, x.shape[1]))  # (steps*n, 1)
    y_sum = y.sum()
    y_sum.backward()

    grads = x_interp.grad.reshape(steps, x.shape[0], x.shape[1])  # (steps, n, d)
    avg_grads = grads.mean(dim=0)  # (n, d)

    attributions = (x - baseline) * avg_grads
    return attributions

In [6]:
permnos = np.sort(train_strict["permno"].unique())

results = Parallel(n_jobs=N_JOBS, backend="loky")(
    delayed(process_one_permno)(p, train_strict, test, BEST_PARAMS) for p in tqdm(permnos)
)

# split into predictions + xai
pred_rows = []
xai_rows = []
for r in results:
    if r is None:
        continue
    pred_df, xai_df = r
    pred_rows.append(pred_df)
    xai_rows.append(xai_df)

preds = pd.concat(pred_rows, ignore_index=True)
preds.sort_values(["permno", "date"], inplace=True)

xai_firm = pd.concat(xai_rows, ignore_index=True)
xai_firm.sort_values("permno", inplace=True)

print("Predictions shape:", preds.shape)
print("XAI firm shape:", xai_firm.shape)

100%|██████████| 2716/2716 [01:25<00:00, 31.83it/s]


Predictions shape: (684286, 5)
XAI firm shape: (2716, 8)


In [7]:
mse_ff3 = np.mean((preds[target] - preds["ff3_pred"])**2)
mse_nn  = np.mean((preds[target] - preds["nn_pred"])**2)

print("STRICT TRAINING (excluded tuning val dates) — Test year 2024")
print("FF3 OOS MSE:", mse_ff3)
print("NN  OOS MSE:", mse_nn)
print("Delta (NN - FF3):", mse_nn - mse_ff3)



STRICT TRAINING (excluded tuning val dates) — Test year 2024
FF3 OOS MSE: 0.0019203413658183477
NN  OOS MSE: 0.0024240778335206307
Delta (NN - FF3): 0.000503736467702283
